# RLAIF 统一框架(精简版)

> [ch13.ipynb](./ch13.ipynb) 的浓缩版。

## 统一 PO 公式

$$\mathcal{L} = \text{policy\_term}(r_t) \cdot A_t - \beta \cdot \text{KL}$$

| 算法 | policy_term | advantage | 模型数 |
|---|---|---|---|
| PPO | `min(r·A, clip(r,1±ε)·A)` | GAE(Critic) | 4 |
| GRPO | `min(r·A, clip(r,1±ε)·A)` | 组归一化 | 3 |
| CISPO | `clamp(r, max=ε_h)·A` | 组归一化 | 3 |

## GAE vs 组归一化

- **GAE**(PPO):用 Critic 估计 V(s),算 TD 误差 δ,指数衰减累加
- **组归一化**(GRPO/CISPO):同 prompt 生成 6 个回复,`(r - mean) / std`

## 最小 GRPO 代码

```python
# 1. 生成 6 个回复
responses = [model.generate(prompt) for _ in range(6)]
# 2. 打分
rewards = [reward_model(prompt, r) for r in responses]
# 3. 组归一化
advantages = (rewards - mean(rewards)) / (std(rewards) + 1e-4)
# 4. CISPO loss
ratio = exp(new_logps - old_logps)
loss = -(clamp(ratio, max=5.0) * advantages * new_logps).mean()
```